В этом ноутбуке мы разбираем базовый pipeline:
1. Загрузка локальной модели проверки текста (NLI).
2. Формирование claim и evidence.
3. Использование предобученной модели для определения: подтверждает ли evidence утверждение.
   

In [1]:
!pip install transformers torch --quiet

In [2]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
model_name = 'cross-encoder/nli-deberta-v3-small'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)


tokenizer_config.json: 0.00B [00:00, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/568M [00:00<?, ?B/s]

In [3]:
claim = 'The Eiffel Tower is located in Berlin.'
evidence = 'The Eiffel Tower is a landmark in Paris, France.'

In [4]:
def classify_claim(claim, evidence):
    # Encode pair for NLI
    inputs = tokenizer.encode_plus(evidence, claim, return_tensors='pt', truncation=True)
    logits = model(**inputs).logits
    probs = torch.softmax(logits, dim=1).detach().numpy()[0]
    labels = ['entailment', 'neutral', 'contradiction']
    return dict(zip(labels, probs))


## Интерпретация результата
- **entailment** → evidence подтверждает claim.
- **contradiction** → evidence опровергает claim.
- **neutral** → evidence не даёт достаточно информации.

In [ ]:
    result = classify_claim(claim, evidence)
    if result['entailment'] > 0.5:
      print('Вердикт: подтверждено')
    elif result['contradiction'] > 0.5:
      print('Вердикт: опровергнуто')
    else:
      print('Вердикт: недостаточно данных')
    print('Scores:', result)

In [6]:
examples = [
            ("Water boils at 90 degrees Celsius at sea level.", "At sea level, water boils at 100°C."),
            ("Mars is larger than Earth.", "Earth has a diameter of ~12,700 km, while Mars is smaller at ~6,800 km."),
            ("Shakespeare wrote 'The Odyssey'.", "The Odyssey was written by Homer, not Shakespeare.")
]

In [19]:
for claim, evidence in examples:
  result = classify_claim(claim, evidence)
  print(f'CLAIM: {evidence}')
  print(f'EVIDENCE: {evidence}')
  print('RESULT:', result)
  print('-'*60)

CLAIM: At sea level, water boils at 100°C.
EVIDENCE: At sea level, water boils at 100°C.
RESULT: {'entailment': np.float32(0.9705394), 'neutral': np.float32(0.025595447), 'contradiction': np.float32(0.0038650734)}
------------------------------------------------------------
CLAIM: Earth has a diameter of ~12,700 km, while Mars is smaller at ~6,800 km.
EVIDENCE: Earth has a diameter of ~12,700 km, while Mars is smaller at ~6,800 km.
RESULT: {'entailment': np.float32(0.03156008), 'neutral': np.float32(0.95570064), 'contradiction': np.float32(0.012739272)}
------------------------------------------------------------
CLAIM: The Odyssey was written by Homer, not Shakespeare.
EVIDENCE: The Odyssey was written by Homer, not Shakespeare.
RESULT: {'entailment': np.float32(0.99944776), 'neutral': np.float32(0.0003430046), 'contradiction': np.float32(0.00020922876)}
------------------------------------------------------------


## Задания
1. Придумайте свои 5 утверждений и подберите к ним evidence
- Два должны быть правдивыми, два ложными, одно — нейтральным.
- Протестируйте их с помощью модели.
- Объясните, почему модель выдала такой результат.

2. Измените wording evidence так, чтобы модель путалась. Например: используйте более расплывчатые формулировки и посмотрите, растёт ли вероятность *neutral*.

3. Создайте mini‑dataset из 10 пар (claim, evidence) и посчитайте accuracy модели
- Разделите вручную на entailment / contradiction / neutral.
- Напишите код для подсчёта доли правильных классификаций.

4. Усложнённое задание: многошаговые доказательства
- Придумайте example, где доказательство состоит из **2–3 предложений**, и проверьте, как модель справляется.

5. Найдите модель для русского языка и проделайте для неё задания выше.
   